# Tutorial: SI-ConvNeXt for isothermal dendrite growth

A lightweight demo on how to build, train, and roll out the autoregressive deep surrogate (ADS) of

> Ji et al., *Scalable Autoregressive Deep Surrogates
> for Dendritic Microstructure Dynamics*, arXiv:2511.03884 (2025),
> https://arxiv.org/abs/2511.03884

- ## Section 1: File structure
- ## Section 2: Build the SI-ConvNeXt model
- ## Section 3: Training (next-step prediction)
- ## Section 4: Autoregressive rollout from a trained checkpoint
- ## Section 5: Visualization and validation

No data is included; cells that need data print `OK`/`MISS` and skip if files are absent.

# Section 1: File structure

- **NPS package**: `pip install -e .` from <https://github.com/llnl/NPS>, or set `NPS_PATH` in Section 2.
- **Dataset (`data/isothermal_training.npy`)**: user-supplied, shape `(N_seq, N_t, Ny, Nx, C) = (20, 250, 64, 64, 2)`, channels $(\phi, U)$; $64 \times 64$ grid from $4 \times 4$ block averaging.
- **Checkpoints (`checkpoints/`)**: trained weights (written by Section 3, read by Section 4).
- **Outputs**: `rollout_isothermal.npy`, `mse_isothermal.svg`.

# Section 2: Build the SI-ConvNeXt model

Isotropic ConvNeXt blocks at a single spatial resolution (no downsampling), enabling zero-shot spatial extrapolation. Channels $(\phi, U)$, depth $M = 7$, hidden dim 64, kernel 3, **periodic** padding.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch

# Point NPS_PATH at your local clone of github.com/llnl/NPS, OR
# leave as None if NPS is already importable.
NPS_PATH = None  # e.g., Path('/path/to/NPS')
if NPS_PATH is not None:
    sys.path.insert(0, str(Path(NPS_PATH).resolve()))

from NPS.model.convnext import ConvNeXtIsotropic

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch =', torch.__version__, ' device =', DEVICE)


In [ ]:
model = ConvNeXtIsotropic(
    in_chans=2, num_classes=2,
    depth=7, dim=64, kernel_size=3,
    DIM=2, periodic=True,
).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'SI-ConvNeXt params: {n_params:,}')


# Section 3: Training (next-step prediction)

Predict frame $n+1$ from frame $n$: 1000 epochs, AdamW. Minimal driver; the manuscript additionally uses $4m$ point-group augmentation and Gaussian input noise ($\sigma = 0.001$).

In [ ]:
def next_step_loss(model, frame_n, frame_np1):
    """MSE loss for predicting frame_np1 from frame_n; tensors (B, 2, Ny, Nx)."""
    pred = model(frame_n)
    return torch.mean((pred - frame_np1) ** 2)

def train_one_step(model, optim, frame_n, frame_np1):
    """Single AdamW gradient step."""
    model.train()
    loss = next_step_loss(model, frame_n, frame_np1)
    optim.zero_grad(); loss.backward(); optim.step()
    return float(loss)

In [ ]:
import os

DATA_PATH = 'data/isothermal_training.npy'           # (N_seq, N_t, 64, 64, 2), see Section 1
CKPT_PATH = 'checkpoints/isothermal_si_convnext.pt'

for f in [DATA_PATH]:
    print(('OK  ' if os.path.exists(f) else 'MISS'), f)

if os.path.exists(DATA_PATH):
    train = np.load(DATA_PATH, mmap_mode='r')        # (N_seq, N_t, Ny, Nx, 2)
    N_seq, N_t = train.shape[:2]
    pairs = [(s, t) for s in range(N_seq) for t in range(N_t - 1)]
    optim = torch.optim.AdamW(model.parameters(), lr=8e-4)   # manuscript sweeps 2e-4 to 8e-4, keeps validation-best
    batch = 16                                       # adjust to your memory budget

    for epoch in range(1000):
        np.random.shuffle(pairs)
        running = 0.0
        for i in range(0, len(pairs), batch):
            s = np.array([p[0] for p in pairs[i:i + batch]])
            t = np.array([p[1] for p in pairs[i:i + batch]])
            frame_n   = torch.from_numpy(np.asarray(train[s, t],     dtype=np.float32)).permute(0, 3, 1, 2).to(DEVICE)
            frame_np1 = torch.from_numpy(np.asarray(train[s, t + 1], dtype=np.float32)).permute(0, 3, 1, 2).to(DEVICE)
            running += train_one_step(model, optim, frame_n, frame_np1) * len(s)
        if epoch % 50 == 0:
            print(f'epoch {epoch:4d}  mean next-step MSE {running / len(pairs):.3e}')

    os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
    torch.save(model.state_dict(), CKPT_PATH)
    print('saved', CKPT_PATH)
else:
    print('Dataset missing -- skipping training. See Section 1 for the expected layout.')

# Section 4: Autoregressive rollout from a trained checkpoint

Feed the model's output back as the next input. The rollout below matches the stored ground-truth length so Section 5 can compare frame by frame.

In [ ]:
@torch.no_grad()
def autoregressive_rollout(model, ic, n_steps, device=None):
    """Roll out for n_steps from channel-last frame ic (Ny, Nx, C).

    Returns (n_steps + 1, Ny, Nx, C); index 0 is the IC.
    """
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    state = (torch.from_numpy(ic.astype(np.float32))
             .permute(2, 0, 1).unsqueeze(0).to(device))
    out = [state.cpu().squeeze(0).permute(1, 2, 0).numpy()]
    for _ in range(n_steps):
        state = model(state)
        out.append(state.cpu().squeeze(0).permute(1, 2, 0).numpy())
    return np.stack(out, axis=0)

In [ ]:
import os

DATA_PATH = 'data/isothermal_training.npy'
CKPT_PATH = 'checkpoints/isothermal_si_convnext.pt'

for f in [DATA_PATH, CKPT_PATH]:
    print(('OK  ' if os.path.exists(f) else 'MISS'), f)

if os.path.exists(DATA_PATH) and os.path.exists(CKPT_PATH):
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    gt = np.load(DATA_PATH, mmap_mode='r')[0]        # first trajectory, (N_t, Ny, Nx, 2)
    rollout = autoregressive_rollout(model, np.asarray(gt[0]), n_steps=len(gt) - 1)
    np.save('rollout_isothermal.npy', rollout)
    print('rollout shape:', rollout.shape, ' -> saved rollout_isothermal.npy')
else:
    print('Need the dataset and a trained checkpoint (Section 3) to roll out.')

# Section 5: Visualization and validation

### Section 5.1: Ground truth (GT) vs ADS prediction at selected steps

In [ ]:
%matplotlib inline
import os
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = 'data/isothermal_training.npy'
ROLLOUT_PATH = 'rollout_isothermal.npy'

for f in [DATA_PATH, ROLLOUT_PATH]:
    print(('OK  ' if os.path.exists(f) else 'MISS'), f)

if os.path.exists(DATA_PATH) and os.path.exists(ROLLOUT_PATH):
    gt  = np.load(DATA_PATH, mmap_mode='r')[0]       # (N_t, Ny, Nx, 2)
    ads = np.load(ROLLOUT_PATH)                      # (n_steps + 1, Ny, Nx, 2)
    timesteps = [0, 50, 150, len(gt) - 1]

    fig, axes = plt.subplots(len(timesteps), 2, figsize=(6.5, 3.2 * len(timesteps)))
    for i, t in enumerate(timesteps):
        axes[i, 0].imshow(gt[t, ..., 0], interpolation='nearest')
        axes[i, 0].set_title(rf'GT $\phi$, step {t}')
        axes[i, 1].imshow(ads[t, ..., 0], interpolation='nearest')
        axes[i, 1].set_title(rf'ADS $\phi$, step {t}')
        for ax in axes[i]:
            ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Run Sections 3-4 first (or drop in your own rollout).')

### Section 5.2: Validation with per-step normalized MSE

$\mathrm{MSE}^i(t)$ normalized by the squared initial-field range $r_i^2$, as defined in the manuscript Methods.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = 'data/isothermal_training.npy'
ROLLOUT_PATH = 'rollout_isothermal.npy'

for f in [DATA_PATH, ROLLOUT_PATH]:
    print(('OK  ' if os.path.exists(f) else 'MISS'), f)

if os.path.exists(DATA_PATH) and os.path.exists(ROLLOUT_PATH):
    gt  = np.load(DATA_PATH, mmap_mode='r')[0]
    ads = np.load(ROLLOUT_PATH)
    n = min(len(gt), len(ads))
    gt0 = np.asarray(gt[0], dtype=np.float64).reshape(-1, gt.shape[-1])
    r = gt0.max(0) - gt0.min(0)                      # initial-field range r_i, per channel
    mse = ((np.asarray(gt[:n], dtype=np.float64) - ads[:n]) ** 2).mean(axis=(1, 2)) / r**2

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(mse[:, 0], label=r'$\phi$')
    ax.plot(mse[:, 1], label=r'$U$')
    ax.set_xlabel('Rollout step')
    ax.set_ylabel('Normalized MSE')
    ax.set_yscale('log')
    ax.legend(loc='best', frameon=False)
    fig.savefig('mse_isothermal.svg', format='svg', bbox_inches='tight', pad_inches=0.1)
    plt.show()
else:
    print('Run Sections 3-4 first (or drop in your own rollout).')